# Curso APIs y Web Scraping — Demo en vivo### Capítulo 1: Introducción a APIs · Capítulo 2: Trabajar con datos JSONNotebook para seguir la clase paso a paso. Ejecuta cada celda con **Shift + Enter**.> Funciona igual en Jupyter, VS Code o Google Colab. No necesitas ninguna API key.

---## 0. Preparación

In [ ]:
# En Colab, requests ya viene instalado. Si estás en local y falla, descomenta:# !pip install requests pandasimport jsonimport requestsprint("requests", requests.__version__)

---# Capítulo 1 · Sesión 2 · Bloque 2## 1. Tu primera petición**Analogía:** le pedimos al camarero (la API) un plato (los datos).

In [ ]:
r = requests.get("https://dummyjson.com/products/1", timeout=10)r

`r` **NO** son los datos: es el objeto Respuesta completo. Vamos a inspeccionarlo.

In [ ]:
print("Status code :", r.status_code)   # 200 = todo bienprint("¿OK?        :", r.ok)print("Tiempo      :", r.elapsed.total_seconds(), "segundos")print("URL         :", r.url)print("Content-Type:", r.headers["Content-Type"])

### `.text` vs `.json()` — la diferencia clave

In [ ]:
texto = r.text          # la respuesta tal cual llegó: TEXTOprint(type(texto))print(texto[:150], "...")

In [ ]:
datos = r.json()        # ese texto convertido a estructuras de Pythonprint(type(datos))      # <class 'dict'>  ← ¡un diccionario!datos

In [ ]:
# Por eso esto falla:  r["title"]   →  TypeError# Y esto funciona:print(datos["title"], "→ $", datos["price"])

---# Capítulo 1 · Sesión 2 · Bloque 1## 2. Los métodos HTTP| Verbo | Acción | Analogía de la biblioteca ||---|---|---|| GET | leer | pedir un libro prestado || POST | crear | donar un libro nuevo || PUT | reemplazar | cambiar un libro dañado || DELETE | eliminar | retirar del catálogo |

In [ ]:
BASE = "https://dummyjson.com"# GET — pedir datos (no modifica nada)r = requests.get(f"{BASE}/products/1", timeout=10)print("GET   ", r.status_code, "→", r.json()["title"])

In [ ]:
# POST — crear un recurso nuevo (los datos van en el CUERPO, con json={...})r = requests.post(f"{BASE}/products/add",                  json={"title": "Laptop BSG", "price": 1200},                  timeout=10)print("POST  ", r.status_code, "→ id asignado:", r.json()["id"])

In [ ]:
# PUT — actualizarr = requests.put(f"{BASE}/products/1", json={"price": 999}, timeout=10)print("PUT   ", r.status_code, "→", r.json()["price"])# DELETE — eliminar (sin body)r = requests.delete(f"{BASE}/products/1", timeout=10)print("DELETE", r.status_code, "→ isDeleted:", r.json()["isDeleted"])

### Query params: nunca los pegues a mano`params={...}` deja que requests arme la URL, codificando espacios y tildes correctamente.

In [ ]:
r = requests.get(f"{BASE}/products/search",                 params={"q": "laptop gamer", "limit": 3},                 timeout=10)print("URL construida:", r.url)   # fíjate en el %20for p in r.json()["products"]:    print(" ·", p["title"], "→ $", p["price"])

## 3. Códigos de estado- **2xx** éxito · **3xx** redirección- **4xx** la culpa es TUYA · **5xx** la culpa es del SERVIDOR`401` = "no sé quién eres" · `403` = "sé quién eres y no pasas"

In [ ]:
for url, desc in [    ("https://dummyjson.com/products/1",      "existe"),    ("https://dummyjson.com/products/999999", "no existe"),]:    r = requests.get(url, timeout=10)    print(f"{r.status_code}  {desc}")

In [ ]:
# raise_for_status() convierte el error HTTP en una excepción de Pythontry:    r = requests.get("https://dummyjson.com/products/999999", timeout=10)    r.raise_for_status()    print(r.json()["title"])except requests.exceptions.HTTPError as e:    print("HTTPError capturado:", e)

---# Capítulo 2 · Sesión 1## 4. Fundamentos de JSON**El truco para no confundirse:**- Con la **s** → trabaja con **S**trings: `dumps` / `loads`- Sin la **s** → trabaja con **f**iles: `dump` / `load`

In [ ]:
usuario = {    "nombre": "Ana Torres",    "edad": 30,    "activo": True,          # Python: mayúscula    "cupon": None,           # Python: None    "roles": ["admin", "editor"],    "perfil": {"pais": "Perú", "idioma": "es"},}texto_json = json.dumps(usuario, indent=2, ensure_ascii=False)print(texto_json)

In [ ]:
# Sin ensure_ascii=False las tildes se escapan:print(json.dumps({"pais": "Perú"}))print(json.dumps({"pais": "Perú"}, ensure_ascii=False))

In [ ]:
# loads: de vuelta a Pythonrecibido = '{"producto": "Laptop", "precio": 3200.5, "activo": true, "cupon": null}'d = json.loads(recibido)for k, v in d.items():    print(f"{k:<10} = {str(v):<12} ({type(v).__name__})")# JSON true → Python True    |    JSON null → Python None

### Los 4 errores de sintaxis más comunes

In [ ]:
malos = [    ("{'nombre': 'Ana'}",     "comillas simples"),    ('{"activo": True}',      "True en mayúscula"),    ('{"lista": [1, 2, 3,]}', "coma final"),    ('{"a": 1} // nota',      "comentarios"),]for texto, motivo in malos:    try:        json.loads(texto)    except json.JSONDecodeError as e:        print(f"{motivo:<22} → {e.msg}")

---# Capítulo 2 · Sesión 2 · Bloque 1## 5. Estructuras anidadas### 🔑 LA REGLA DE ORO- Corchetes con **texto** → entras a un **diccionario** (buscas una clave)- Corchetes con **número** → entras a una **lista** (buscas una posición, desde 0)

In [ ]:
pedido = {    "pedido_id": 1043,    "cliente": {        "nombre": "Ana Torres",        "documento": {"tipo": "DNI", "numero": "45871236"},    },    "items": [        {"producto": "Laptop", "cantidad": 1, "precio": 3200.0},        {"producto": "Mouse",  "cantidad": 2, "precio": 45.5},    ],    "pagado": True,}print(pedido["cliente"]["nombre"])                 # dict → dictprint(pedido["cliente"]["documento"]["numero"])    # dict → dict → dictprint(pedido["items"][1]["producto"])              # dict → list → dict

**Ejercicio relámpago:** ¿cómo obtienes el precio del mouse? ¿Y cuántos productos tiene el pedido?

In [ ]:
total = 0for item in pedido["items"]:    subtotal = item["cantidad"] * item["precio"]    total += subtotal    print(f"{item['producto']:<10} x{item['cantidad']}  =  {subtotal:>8.2f}")print(f"{'TOTAL':<14}  =  {total:>8.2f}")

### Acceso seguro con `.get()`En APIs reales los campos opcionales faltan a cada rato. `.get()` evita que tu script muera en el registro 347 de 1000.

In [ ]:
try:    pedido["descuento"]except KeyError as e:    print("KeyError:", e)print("Con .get():", pedido.get("descuento"))print("Con default:", pedido.get("descuento", 0))

## 6. El patrón de 5 pasos contra una API real1. **Pedir** · 2. **Verificar** · 3. **Convertir** · 4. **Explorar** · 5. **Recorrer**

In [ ]:
r = requests.get("https://dummyjson.com/products", params={"limit": 5}, timeout=10)  # 1r.raise_for_status()                                                                   # 2data = r.json()                                                                        # 3print("type :", type(data))          # 4 — explorar antes de asumirprint("keys :", list(data.keys()))print("total:", data["total"])

In [ ]:
for p in data["products"]:                                                             # 5    print(f"{p['id']:>3}. {p['title']:<38} $ {p['price']:<8} [{p.get('brand','sin marca')}]")

In [ ]:
# Anidamiento profundo: dict → list → dict → list → dictcomentario = data["products"][0]["reviews"][0]["comment"]print(comentario)

---# Capítulo 1 · Sesión 1 · Bloque 3## 7. APIs públicas reales (sin clave)

In [ ]:
# Tipo de cambio USD → PENfx = requests.get("https://open.er-api.com/v6/latest/USD", timeout=10).json()pen = fx["rates"]["PEN"]print(f"1 USD = {pen} PEN   (actualizado: {fx['time_last_update_utc']})")print(f"500 USD = S/ {500 * pen:,.2f}")

In [ ]:
# Clima actual en Limaclima = requests.get(    "https://api.open-meteo.com/v1/forecast",    params={"latitude": -12.0464, "longitude": -77.0428,            "current": "temperature_2m,relative_humidity_2m",            "timezone": "America/Lima"},    timeout=10,).json()actual = clima["current"]print(f"Lima: {actual['temperature_2m']}°C, humedad {actual['relative_humidity_2m']}%")

In [ ]:
# Indicadores de Perú — Banco Mundial# OJO: la respuesta es una LISTA de 2 elementos: [metadatos, [datos]]resp = requests.get("https://api.worldbank.org/v2/country/PER",                    params={"format": "json"}, timeout=10).json()print("type:", type(resp), "| len:", len(resp))   # list de 2 → explora antes de asumirmeta, paises = respperu = paises[0]print("Capital       :", peru["capitalCity"])print("Región        :", peru["region"]["value"].strip())print("Nivel de renta:", peru["incomeLevel"]["value"])print("\nRuta usada: resp[1][0]['region']['value']  →  list → list → dict → dict")

---## 8. Cierre: de la API a un archivo

In [ ]:
r = requests.get("https://dummyjson.com/products", params={"limit": 10}, timeout=10)r.raise_for_status()catalogo = [{"titulo": p["title"], "precio": p["price"]} for p in r.json()["products"]]with open("productos.json", "w", encoding="utf-8") as f:    json.dump(catalogo, f, indent=2, ensure_ascii=False)print(f"Guardados {len(catalogo)} productos en productos.json")print(f"Precio promedio: $ {sum(i['precio'] for i in catalogo) / len(catalogo):.2f}")

In [ ]:
# Bonus con pandasimport pandas as pddf = pd.DataFrame(r.json()["products"])[["id", "title", "category", "price", "rating"]]df.head(10)

---## 🎯 Reto de la sesión1. **Tipo de cambio** — ¿cuántos soles vale 1 dólar? (`rates` → `PEN`)2. **Clima** — temperatura actual en Lima (`current` → `temperature_2m`)3. **Catálogo** — guarda título y precio de 10 productos en `productos.json`**Bonus:** haz un POST a `dummyjson.com/products/add` con `{"title": "Curso BSG", "price": 99}` y revisa el status code.> Los enunciados completos están en `09_reto_clase.py` y las soluciones comentadas en `soluciones/`.

In [ ]:
# Tu código aquí 👇

---## Recordatorio final- **API** = el camarero: le pides por una URL con un verbo, te trae un plato en JSON.- **6 piezas** de una petición: método · URL base · endpoint · params · headers · body.- **Siempre** revisa el status code antes de leer los datos.- **JSON ↔ Python**: `object`→`dict`, `array`→`list`, `true`→`True`, `null`→`None`.- Corchetes con **texto** = clave · corchetes con **número** = posición.**Próxima sesión:** Web Scraping — qué hacer cuando NO hay API.